# RAG baseline vs cross-encoder re-ranking

## 1. Config
All settings live in one cell. Set the environment variable `DOC` to `nfl` or `nba` to pick the rulebook. Both pipelines share the same chunks, retrieval depth (`TOP_K`) and prompt.

In [1]:
import os, re, time, random, statistics, urllib.request
import numpy as np, pandas as pd, torch, faiss, pymupdf
from sentence_transformers import SentenceTransformer, CrossEncoder
from openai import OpenAI

DOC = os.getenv("DOC", "nfl")   # "nfl" or "nba"
PDF_URLS = {
    "nfl": "https://media.nfl.com/content/dam/communications/Official%20Playing%20Rules/2026%20Official%20Playing%20Rules.pdf",
    "nba": "https://cdn.nba.com/manage/2026/01/Official-2025-26-NBA-Playing-Rules.pdf",
}
DOC_NAMES = {"nfl": "NFL 2026 Official Playing Rules", "nba": "NBA 2025-26 Official Playing Rules"}

EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"   # bi-encoder (stage 1)
RERANK_MODEL = "cross-encoder/ettin-reranker-1b-v1"       # cross-encoder (stage 2)
LLM_MODEL    = "gemini-3.5-flash-lite"                    # Gemini API (needs GOOGLE_API_KEY in .env)
LLM_URL      = "https://generativelanguage.googleapis.com/v1beta/openai/"   # Gemini's OpenAI-compatible endpoint

TOP_K, TOP_N = 10, 3             # retrieve 10, send 3 to the LLM
CHUNK_WORDS, OVERLAP = 120, 30
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
SEED = 42

# Seed every RNG we use so reruns are repeatable
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Load GOOGLE_API_KEY from .env if the file exists (the key is never printed)
if os.path.exists(".env"):
    for line in open(".env"):
        k, _, v = line.strip().partition("=")
        if k and not k.startswith("#") and v:
            os.environ.setdefault(k, v.strip().strip("'").strip('"'))
print(DOC_NAMES[DOC], "| device:", DEVICE)

/Users/va/LMA A1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NFL 2026 Official Playing Rules | device: mps


## 2. Load PDF
Download the rulebook to `data/` if it is not already there, then extract the text of each page with PyMuPDF and collapse all whitespace runs to single spaces.

In [2]:
os.makedirs("data", exist_ok=True)
pdf_path = f"data/{DOC}.pdf"

# Download once; a browser-like User-Agent avoids 403s from the CDNs
if not os.path.exists(pdf_path):
    req = urllib.request.Request(PDF_URLS[DOC], headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, open(pdf_path, "wb") as f:
        f.write(resp.read())

# One cleaned string per page (list index 0 = page 1)
with pymupdf.open(pdf_path) as pdf:
    pages = [re.sub(r"\s+", " ", page.get_text()).strip() for page in pdf]
print(f"{len(pages)} pages, {sum(len(p.split()) for p in pages):,} words")

245 pages, 200,753 words


## 3. Chunk
Split each page into fixed windows of `CHUNK_WORDS` words with `OVERLAP` words shared between neighbours; windows never cross a page boundary, so every chunk has one page number. The sanity cell checks how many chunks exceed MiniLM's 256-token input limit (longer text is silently truncated).

In [3]:
chunks = []
step = CHUNK_WORDS - OVERLAP
for page_no, text in enumerate(pages, start=1):
    words = text.split()
    for start in range(0, len(words), step):
        window = words[start:start + CHUNK_WORDS]
        chunks.append({"id": len(chunks), "page": page_no, "text": " ".join(window)})
        if start + CHUNK_WORDS >= len(words):   # window reached page end, stop (avoids a tiny duplicate tail chunk)
            break
print("chunks:", len(chunks))

chunks: 2274


In [4]:
# Sanity check: MiniLM truncates input at 256 tokens, so few chunks should be longer than that
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(EMBED_MODEL)
n_tokens = [len(ids) for ids in tok([c["text"] for c in chunks], add_special_tokens=True)["input_ids"]]
pct_trunc = 100 * np.mean(np.array(n_tokens) > 256)
print(f"chunks: {len(chunks)} | max tokens: {max(n_tokens)} | median: {int(np.median(n_tokens))} | over 256 tokens: {pct_trunc:.2f}%")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1392 > 512). Running this sequence through the model will result in indexing errors


chunks: 2274 | max tokens: 1441 | median: 163 | over 256 tokens: 0.66%


## 4. Index
Embed every chunk with the MiniLM bi-encoder and store the vectors in an exact FAISS flat index. The vectors are L2-normalised, so inner product equals cosine similarity. The Ettin cross-encoder is also loaded in this cell, but it is not part of the index. A cross-encoder needs the query and the chunk together, so it can only score (query, chunk) pairs at query time, inside `rerank`.

In [5]:
bi = SentenceTransformer(EMBED_MODEL, device=DEVICE)
texts = [c["text"] for c in chunks]
emb = bi.encode(texts, normalize_embeddings=True, batch_size=64)   # shape (n_chunks, 384)

index = faiss.IndexFlatIP(emb.shape[1])   # exact inner-product search
index.add(emb.astype("float32"))
print("indexed vectors:", index.ntotal)

ce = CrossEncoder(RERANK_MODEL, device=DEVICE)   # scores (query, chunk) pairs jointly

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14019.58it/s]

indexed vectors: 2274


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 170/170 [00:00<00:00, 7543.07it/s]

## 5. Core functions
`retrieve` does the FAISS top-k search, `rerank` re-scores those candidates with the cross-encoder, and `generate` asks Gemini using one shared prompt. `run` executes both pipelines on a query: Pipeline 1 sends the bi-encoder top-3 to the LLM, Pipeline 2 sends the cross-encoder top-3 of the same 10 candidates.

In [6]:
PROMPT = """You are answering questions about the {doc_name}.
Use ONLY the excerpts below. If they do not contain the answer, say "Not covered in the provided excerpts."
Cite the page number(s) you used.

Excerpts:
{context}

Question: {question}
Answer:"""

assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env (see README)"
llm = OpenAI(base_url=LLM_URL, api_key=os.environ["GOOGLE_API_KEY"], max_retries=5)   # retries also cover rate limits


def retrieve(q, k=TOP_K):
    """Bi-encoder stage: embed the query, return top-k chunk ids and cosine scores."""
    qv = bi.encode([q], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(qv, k)
    return ids[0].tolist(), scores[0].tolist()


def rerank(q, ids):
    """Cross-encoder stage: score each (query, chunk) pair, return ids sorted by score (high to low)."""
    scores = ce.predict([(q, chunks[i]["text"]) for i in ids])
    order = np.argsort(-scores)
    return [ids[j] for j in order], [float(scores[j]) for j in order]


def build_prompt(q, ids):
    """Fill the shared prompt with the selected chunks, each tagged with its page number."""
    context = "\n\n".join(f"[p. {chunks[i]['page']}] {chunks[i]['text']}" for i in ids)
    return PROMPT.format(doc_name=DOC_NAMES[DOC], context=context, question=q)


def generate(q, ids):
    """Ask Gemini with the shared prompt (temperature 0)."""
    resp = llm.chat.completions.create(
        model=LLM_MODEL, messages=[{"role": "user", "content": build_prompt(q, ids)}],
        temperature=0, max_tokens=2048,
    )
    return (resp.choices[0].message.content or "").strip()


def run(q):
    """Run both pipelines on one query and time every stage (seconds)."""
    out = {}
    t = time.perf_counter()
    out["base_ids"], out["base_scores"] = retrieve(q)
    out["t_retrieve"] = time.perf_counter() - t

    t = time.perf_counter()
    out["rr_ids"], out["rr_scores"] = rerank(q, out["base_ids"])
    out["t_rerank"] = time.perf_counter() - t

    t = time.perf_counter()
    out["answer1"] = generate(q, out["base_ids"][:TOP_N])   # Pipeline 1: baseline top-3
    out["t_llm1"] = time.perf_counter() - t

    t = time.perf_counter()
    out["answer2"] = generate(q, out["rr_ids"][:TOP_N])     # Pipeline 2: re-ranked top-3
    out["t_llm2"] = time.perf_counter() - t
    return out

## 6. Probe helper
`show` prints the retrieved top-10 for a query with bi-encoder and cross-encoder ranks and scores, without calling the LLM. A chunk counts as gold if any gold snippet appears in its text (case-insensitive, whitespace-normalised); `gold_ranks` returns the best gold rank under each ordering.

In [7]:
def norm(s):
    """Lowercase and collapse whitespace so snippet matching ignores formatting."""
    return re.sub(r"\s+", " ", s).lower().strip()


def is_gold(chunk_id, gold):
    text = norm(chunks[chunk_id]["text"])
    return any(norm(g) in text for g in (gold or []))


def show(q, gold=None):
    """Top-10 table: bi-encoder vs cross-encoder rank/score, with a gold flag. No LLM call."""
    base_ids, base_scores = retrieve(q)
    rr_ids, rr_scores = rerank(q, base_ids)
    ce_rank = {cid: r for r, cid in enumerate(rr_ids, start=1)}
    ce_score = dict(zip(rr_ids, rr_scores))
    rows = [{
        "chunk": cid,
        "page": chunks[cid]["page"],
        "bi_rank": r,
        "bi_score": round(s, 3),
        "ce_rank": ce_rank[cid],
        "ce_score": round(ce_score[cid], 3),
        "text": chunks[cid]["text"][:90],
        "gold": is_gold(cid, gold),
    } for r, (cid, s) in enumerate(zip(base_ids, base_scores), start=1)]
    return pd.DataFrame(rows)


def gold_ranks(q, gold):
    """Best (lowest) rank of any gold chunk in the bi-encoder and cross-encoder orderings; None if not in the top-10."""
    base_ids, _ = retrieve(q)
    rr_ids, _ = rerank(q, base_ids)
    first = lambda ids: next((r for r, cid in enumerate(ids, start=1) if is_gold(cid, gold)), None)
    return first(base_ids), first(rr_ids)

## 7. Failure set
Each query below has a gold snippet copied from the rulebook text. `show` prints the top-10 for every query, `run` executes both pipelines once per query, and the tables report where the gold chunk sits under each ranking and what each pipeline answered.

In [8]:
# Queries with gold snippets (substrings of the real chunk text). "control" and "CE did not help" rows are kept on purpose.
FAILURE_SETS = {
    "nfl": [
        {"query": "Can a coach challenge a play that begins after the two-minute warning?", "category": "exception",
         "gold": ["Only the Replay Official or the Senior Vice President of Officiating or his or her designee may initiate a review"]},
        # Distractor: the baseline top 3 are approved-ruling (A.R.) casebook chunks (p. 142, p. 194) that share "fumble" and
        # "two-minute warning" with the query; the rule itself (Article 6, p. 42) is at baseline rank 10.
        {"query": "Who is allowed to advance a fumble after the two-minute warning?", "category": "keyword trap / distractor",
         "gold": ["The player who fumbled is the only player of his team who is permitted to recover and advance the ball"]},
        {"query": "How many challenges does a team get?", "category": "buried",
         "gold": ["A team will be permitted a third challenge if it is successful on at least one of its challenges"]},
        {"query": "How long is the break between overtime periods?", "category": "buried",
         "gold": ["Between each overtime period, there shall be a two-minute intermission"]},
        {"query": "Can a team use two charged timeouts back to back?", "category": "negation",
         "gold": ["a second charged team timeout by either team during the same dead ball period is prohibited"]},
        {"query": "Can a kicker wear a detachable kicking toe?", "category": "negation",
         "gold": ["A detachable kicking toe is prohibited"]},
        {"query": "Which offensive players are not allowed to catch a forward pass?", "category": "control (already top 3)",
         "gold": ["are ineligible to catch a legal or illegal forward pass thrown from behind the line of scrimmage"]},
    ],
    "nba": [
        {"query": "Can a player be granted a timeout while airborne?", "category": "negation",
         "gold": ["A player shall not be granted any timeout if both of his feet are in the air"]},
        {"query": "Does the shot clock reset to 14 or 24 after a foul?", "category": "keyword trap",
         "gold": ["or stay the same or reset to 14, whichever is greater"]},
        {"query": "Can a coach walk past half court?", "category": "negation",
         "gold": ["A coach is not permitted to cross the midcourt line"]},
        {"query": "Can a substitute replace the player shooting free throws?", "category": "exception",
         "gold": ["shall not replace a free throw shooter or a player involved in a jump ball unless dictated to do so by an injury or ejection"]},
        {"query": "What are the mandatory timeouts in each period?", "category": "buried",
         "gold": ["There must be two mandatory timeouts in each period"]},
        {"query": "What makes a shot attempt legal for the shot clock?", "category": "buried",
         "gold": ["the ball must make contact with the basket ring"]},
        {"query": "Does an injured player's free throw get taken by the coach's choice of substitute?", "category": "CE did not help",
         "gold": ["his coach shall designate an eligible substitute from the bench"]},
    ],
}
FAILURE = FAILURE_SETS[DOC]
print(len(FAILURE), "queries for", DOC)

7 queries for nfl


In [9]:
# (a) Top-10 for every query: bi-encoder vs cross-encoder rank and score, gold chunk flagged (no LLM call)
from IPython.display import display
for item in FAILURE:
    print(f"\n=== [{item['category']}] {item['query']}")
    display(show(item["query"], item["gold"]))


=== [exception] Can a coach challenge a play that begins after the two-minute warning?


,chunk,page,bi_rank,bi_score,ce_rank,ce_score,text,gold
0,2190,237,1,0.674,2,9.436,the two-minute warning. During the two-minute ...,False
1,668,70,2,0.658,3,8.449,for plays that begin after the two-minute warn...,False
2,814,99,3,0.657,7,7.746,0:05 and start on the snap. A.R. 4.14 GAME AND...,False
3,134,20,4,0.652,10,6.350,the two-minute warning: Loss of five yards fro...,False
4,2197,237,5,0.648,6,8.306,the clock reaches 2:00. If the two-minute warn...,False
5,2196,237,6,0.635,4,8.391,downed with 1:52 remaining on the game clock. ...,False
6,632,67,7,0.629,1,10.241,its timeouts: Loss of 15 yards enforced as a f...,True
7,669,70,8,0.624,9,6.774,can avoid the 10-second runoff by taking a cha...,False
8,896,106,9,0.613,8,7.256,ready for play. Team A has committed a foul th...,False
9,662,70,10,0.613,5,8.333,that the clock should have stopped for a team ...,False



=== [keyword trap / distractor] Who is allowed to advance a fumble after the two-minute warning?


,chunk,page,bi_rank,bi_score,ce_rank,ce_score,text,gold
0,1257,142,1,0.761,4,8.970,"ground, this is considered a fumble by A1 on t...",False
1,1258,142,2,0.733,3,9.392,"Before the two-minute warning, only Team A is ...",False
2,1778,194,3,0.731,10,6.805,"the two-minute warning, with the clock running...",False
3,807,98,4,0.717,8,7.253,back into the field before it is spotted at th...,False
4,2038,222,5,0.716,9,7.014,"two-minute warning, A1 runs to the B12 and fum...",False
5,804,98,6,0.713,5,8.708,"has been compromised in some way, there will b...",False
6,1038,120,7,0.710,7,7.397,"two-minute warning, the ball returns to the sp...",False
7,2089,227,8,0.708,6,8.271,recovery to the team that recovers the ball in...,False
8,371,41,9,0.704,2,9.737,action that conserves time (4-7-1). Penalty: F...,False
9,379,42,10,0.704,1,10.308,"the fumble, or the spot of the recovery if the...",True



=== [buried] How many challenges does a team get?


...

,chunk,page,bi_rank,bi_score,ce_rank,ce_score,text,gold
0,670,70,1,0.716,5,9.003,team timeout and then unsuccessfully challenge...,False
1,2213,239,2,0.669,3,9.487,TIMEOUTS If a team challenges and loses the ch...,False
2,126,19,3,0.662,2,10.109,exhausted his permitted timeouts in a half or ...,False
3,125,19,4,0.646,4,9.079,shall suspend play while the ball is dead and ...,False
4,132,20,5,0.628,6,8.538,"used its three charged team timeouts, the team...",False
5,127,19,6,0.625,1,10.867,team may be granted a charged team timeout dur...,True
6,631,67,7,0.605,10,7.072,teams can benefit from the review. (c) The hea...,False
7,103,16,8,0.602,7,8.387,"entering the field, unless they are incoming s...",False
8,124,19,9,0.600,9,7.649,or behind a goal line; (e) at the end of a dow...,False
9,847,102,10,0.598,8,7.995,off the field. Ruling: First-and-10 on B25. Te...,False



=== [negation] Can a kicker wear a detachable kicking toe?


,chunk,page,bi_rank,bi_score,ce_rank,ce_score,text,gold
0,212,26,1,0.640,3,7.616,artiﬁcial limb on his kicking leg must have a ...,False
1,211,26,2,0.613,4,7.327,"appropriate use of tape, players will be allow...",False
2,209,26,3,0.601,6,6.304,of the lower leg and ankle area due to imprope...,False
3,528,57,4,0.548,9,3.079,and one of them runs into the kicker or holder...,False
4,232,28,5,0.545,7,5.152,game. Notes: (1) If a player is suspended for ...,False
5,214,27,6,0.535,1,10.029,Item 3. Torn Items. Torn or improperly ﬁtting ...,True
6,230,28,7,0.531,5,7.242,"illegal kicking toe of shoe, an adhesive or sl...",False
7,216,27,8,0.514,8,4.866,"on shoes is permitted, provided it is black or...",False
8,215,27,9,0.504,2,9.861,steel tips are permitted. A detachable kicking...,True
9,678,71,10,0.497,10,2.953,or the side of the opponent’s shoulder pads or...,False



=== [control (already top 3)] Which offensive players are not allowed to catch a forward pass?


,chunk,page,bi_rank,bi_score,ce_rank,ce_score,text,gold
0,332,38,1,0.752,2,9.720,have the numbers of eligible players (0–49 and...,True
1,331,38,2,0.722,1,9.999,a foul for illegal touching. ARTICLE 5. ELIGIB...,False
2,333,38,3,0.699,3,8.863,in Article 5 above are ineligible to catch a l...,True
3,321,37,4,0.688,8,6.366,one forward pass from behind the line during e...,False
4,1167,134,5,0.666,4,7.539,Illegal forward pass. No pass interference can...,False
5,2100,228,6,0.655,9,5.964,"FORWARD/BACKWARD PASSES, ILLEGAL FORWARD PASSE...",False
6,1185,135,7,0.652,5,7.352,"touching of the pass is not a foul, an origina...",False
7,1177,135,8,0.651,6,7.006,"to eligible receiver A2, but ineligible A5 cat...",False
8,789,97,9,0.649,7,6.459,A.R. 3.21 TEAM A FUMBLE—B RECOVERS AND RUNS FO...,False
9,664,70,10,0.649,10,4.852,possession at the spot where possession was lo...,False


In [10]:
# (b) Run both pipelines once per query (includes the two LLM calls) and keep every result
results = {item["query"]: run(item["query"]) for item in FAILURE}
print("done:", len(results), "queries")

done: 7 queries


In [11]:
# (c) Rank-shift table: best gold-chunk rank under the bi-encoder vs under the cross-encoder
rows = []
for item in FAILURE:
    q, out = item["query"], results[item["query"]]
    gold_ids = [c["id"] for c in chunks if is_gold(c["id"], item["gold"])]
    base_rank = next((r for r, cid in enumerate(out["base_ids"], start=1) if cid in gold_ids), None)
    ce_rank = next((r for r, cid in enumerate(out["rr_ids"], start=1) if cid in gold_ids), None)
    rows.append({
        "query": q,
        "category": item["category"],
        "gold_chunks": gold_ids,
        "page": sorted({chunks[i]["page"] for i in gold_ids}),
        "base_rank": base_rank,
        "ce_rank": ce_rank,
        "moved": f"#{base_rank} \u2192 #{ce_rank}" if base_rank else "not in top-10",
        "base_top3": out["base_ids"][:TOP_N],
        "ce_top3": out["rr_ids"][:TOP_N],
    })
rank_shift = pd.DataFrame(rows)
display(rank_shift)

,query,category,gold_chunks,page,base_rank,ce_rank,moved,base_top3,ce_top3
0,Can a coach challenge a play that begins after...,exception,[632],[67],7,1,#7 → #1,"[2190, 668, 814]","[632, 2190, 668]"
1,Who is allowed to advance a fumble after the t...,keyword trap / distractor,[379],[42],10,1,#10 → #1,"[1257, 1258, 1778]","[379, 371, 1258]"
2,How many challenges does a team get?,buried,[632],[67],6,2,#6 → #2,"[631, 743, 669]","[630, 632, 759]"
3,How long is the break between overtime periods?,buried,[684],[72],9,2,#9 → #2,"[686, 685, 679]","[108, 684, 680]"
4,Can a team use two charged timeouts back to back?,negation,[127],[19],6,1,#6 → #1,"[670, 2213, 126]","[127, 126, 2213]"
5,Can a kicker wear a detachable kicking toe?,negation,"[214, 215]",[27],6,1,#6 → #1,"[212, 211, 209]","[214, 215, 212]"
6,Which offensive players are not allowed to cat...,control (already top 3),"[332, 333]",[38],1,2,#1 → #2,"[332, 331, 333]","[331, 332, 333]"


In [12]:
# (d) Both answers per query, printed in full with the chunks each pipeline was given
def label(ids):
    return ", ".join(f"chunk {i} (p. {chunks[i]['page']})" for i in ids[:TOP_N])

for item in FAILURE:
    q, out = item["query"], results[item["query"]]
    print("=" * 100)
    print("Q:", q, f"[{item['category']}]")
    print(f"\nAnswer 1 (baseline top-{TOP_N}: {label(out['base_ids'])}):\n{out['answer1']}")
    print(f"\nAnswer 2 (re-ranked top-{TOP_N}: {label(out['rr_ids'])}):\n{out['answer2']}\n")
    # Gold chunk text (first 2 if overlap gives more) so a reader can check each answer against the rule
    for c in [c for c in chunks if is_gold(c["id"], item["gold"])][:2]:
        print(f"Gold chunk {c['id']} (p. {c['page']}): {c['text']}\n")

Q: Can a coach challenge a play that begins after the two-minute warning? [exception]

Answer 1 (baseline top-3: chunk 2190 (p. 237), chunk 668 (p. 70), chunk 814 (p. 99)):
Based on the provided excerpts, teams can challenge plays "up until the first legal snap or kickoff following the two-minute warning" (p. 237). 

Sources:
- Page 237

Answer 2 (re-ranked top-3: chunk 632 (p. 67), chunk 2190 (p. 237), chunk 668 (p. 70)):
Not covered in the provided excerpts.

Gold chunk 632 (p. 67): its timeouts: Loss of 15 yards enforced as a foul between downs. (e) If a challenge is unsuccessful, the team will be charged a timeout. (f) A team will be permitted a third challenge if it is successful on at least one of its challenges. A fourth challenge will not be permitted. ARTICLE 2. REPLAY OFFICIAL REQUEST FOR REVIEW. Only the Replay Official or the Senior Vice President of Officiating or his or her designee may initiate a review of a play: (a) that begins after the two-minute warning of each half

## 8. Latency profiling
After one warm-up retrieve and rerank, each failure query is timed 10 times for the bi-encoder retrieval and for the cross-encoder re-rank of its top-10 (Table A). Table B uses one measurement per query taken from the section 7 runs, with no new LLM calls. All values are in milliseconds. LLM time depends on answer length, so Table A isolates the cost of re-ranking.

In [13]:
qs = [item["query"] for item in FAILURE]

# Warm-up so one-time MPS/kernel setup is not counted
_ids, _ = retrieve(qs[0]); rerank(qs[0], _ids)

REPS = 10
ret, rr = [], []   # milliseconds, one entry per (query, repetition)
for q in qs:
    for _ in range(REPS):
        t = time.perf_counter(); ids, _ = retrieve(q);  ret.append((time.perf_counter() - t) * 1000)
        t = time.perf_counter(); rerank(q, ids);        rr.append((time.perf_counter() - t) * 1000)
p2_ret = [a + b for a, b in zip(ret, rr)]   # retrieve + rerank from the same repetition

def stats(v):
    v = np.array(v)
    return {"mean_ms": v.mean(), "median_ms": np.median(v), "p95_ms": np.percentile(v, 95)}

# Table A: retrieval stages, 10 reps x every failure query
table_a = pd.DataFrame({
    "Retrieve (embed + FAISS)": stats(ret),
    "Rerank (CE on 10)": stats(rr),
    "Pipeline 1 retrieval": stats(ret),
    "Pipeline 2 retrieval (retrieve + rerank)": stats(p2_ret),
}).T.round(1)
print(f"Table A: retrieval stages (n = {len(ret)} timings)")
display(table_a)

# Table B: generation and end-to-end, one value per query from the section 7 runs
ms = lambda key: [results[q][key] * 1000 for q in qs]   # seconds to ms
llm1, llm2 = ms("t_llm1"), ms("t_llm2")
p1_total = [a + b for a, b in zip(ms("t_retrieve"), llm1)]
p2_total = [a + b + c for a, b, c in zip(ms("t_retrieve"), ms("t_rerank"), llm2)]
table_b = pd.DataFrame({
    "LLM (P1 context)": stats(llm1),
    "LLM (P2 context)": stats(llm2),
    "Pipeline 1 end-to-end (t_retrieve + t_llm1)": stats(p1_total),
    "Pipeline 2 end-to-end (t_retrieve + t_rerank + t_llm2)": stats(p2_total),
}).T.round(1)
print(f"\nTable B: generation and end-to-end (n = {len(qs)} queries)")
display(table_b)
print(f"\ndevice: {DEVICE} | bi-encoder: {EMBED_MODEL} | cross-encoder: {RERANK_MODEL} | LLM: {LLM_MODEL}")

Table A: retrieval stages (n = 70 timings)


,mean_ms,median_ms,p95_ms
Retrieve (embed + FAISS),7.1,7.1,8.1
Rerank (CE on 10),1201.5,1149.2,1444.5
Pipeline 1 retrieval,7.1,7.1,8.1
Pipeline 2 retrieval (retrieve + rerank),1208.6,1156.3,1451.3



Table B: generation and end-to-end (n = 7 queries)


,mean_ms,median_ms,p95_ms
LLM (P1 context),1733.2,1553.8,2894.4
LLM (P2 context),1417.2,1423.3,1650.4
Pipeline 1 end-to-end (t_retrieve + t_llm1),1765.1,1594.4,2912.8
Pipeline 2 end-to-end (t_retrieve + t_rerank + t_llm2),2616.1,2731.7,2806.7



device: mps | bi-encoder: sentence-transformers/all-MiniLM-L6-v2 | cross-encoder: cross-encoder/ettin-reranker-1b-v1 | LLM: gemini-3.5-flash-lite


## 9. Scaling
For one query, time the cross-encoder over all chunks versus over the top-10 candidates, and the FAISS search over the full index. The all-chunk pass is warmed up on the top-10 pairs only and timed over 2 repetitions; the top-10 pass gets one warm-up and 3 repetitions, FAISS one warm-up and 20 (medians reported). The cost of scoring every chunk is then extrapolated linearly to 1,000,000 chunks.

In [14]:
scale_q = "What happens if a defensive player is in the lane for more than three seconds?" if DOC == "nba" \
          else "Who may challenge a play after the two-minute warning?"

def median_ms(fn, reps=3, warmup=None):
    """Warm up once (with `warmup` if given, else `fn`), then return the median wall time of `reps` calls, in milliseconds."""
    (warmup or fn)()
    times = []
    for _ in range(reps):
        t = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t) * 1000)
    return statistics.median(times)

N = len(chunks)
qv = bi.encode([scale_q], normalize_embeddings=True).astype("float32")
top_ids, _ = retrieve(scale_q)

top10_pairs = lambda: ce.predict([(scale_q, chunks[i]["text"]) for i in top_ids])
ce_all_ms = median_ms(lambda: ce.predict([(scale_q, c["text"]) for c in chunks]), reps=2, warmup=top10_pairs)   # every chunk
ce_10_ms  = median_ms(top10_pairs)                                                                              # top-10 only
faiss_ms  = median_ms(lambda: index.search(qv, TOP_K), reps=20)                                       # search over all N vectors

extrap_s = ce_all_ms / N * 1_000_000 / 1000   # linear extrapolation, in seconds
print(pd.DataFrame([{
    "N chunks": N,
    "CE-all ms": round(ce_all_ms, 1),
    "CE-10 ms": round(ce_10_ms, 1),
    "FAISS ms": round(faiss_ms, 3),
    "CE-all @ 1M chunks": f"{extrap_s:,.0f} s (~{extrap_s/3600:.1f} h)",
}]).to_string(index=False))

 N chunks  CE-all ms  CE-10 ms  FAISS ms  CE-all @ 1M chunks
     2274   315314.6    1199.8     0.097 138,661 s (~38.5 h)


## 10. Bi-encoder comparison
This appendix compares candidate bi-encoders on retrieval only, with no cross-encoder and no LLM: each model embeds the same 120-word chunks and ranks all of them for every query in `BI_EVAL[DOC]`. The eval set is the section 7 failure set plus extra NFL probe queries whose gold snippets match at least one chunk. The failure queries were picked because the MiniLM baseline struggled on them, so this set is biased against the baseline.

In [15]:
# Extra NFL probe queries, same format as the failure set. Every gold snippet was checked with is_gold against all chunks and matches at least one.
NFL_PROBES = [
    ("What happens when the kickoff lands short of the landing zone?", "synonym trap",
     ["A kick that lands or touches a player short of the landing zone is treated like a kickoff out of bounds", "lands or is touched short of the landing zone"]),
    ("What contact is a defender allowed to make on a receiver beyond five yards?", "exception",
     ["Beyond the five-yard zone, incidental contact may exist between receiver and defender"]),
    ("Is a hip-drop tackle legal?", "negation",
     ["It is a foul if a player uses the following technique to bring a runner to the ground"]),
    ("Can the defense run into the kicker?", "negation",
     ["run into the kicker before he recovers his balance"]),
    ("How long does the offense have to snap the ball before delay of game?", "buried",
     ["It is a delay of the game if the ball is not put in play by a snap within 40 seconds"]),
    ("Can a player wear long sleeve undergarments with grip sleeves?", "negation",
     ["Players may not wear long-sleeved undergarments that include pebble-grip sleeves"]),
    ("When is the line to gain not ten yards from the snap?", "exception",
     ["except when a goal line is less than 10 yards from this spot"]),
    ("Can a personal foul after the end of a quarter be enforced on the next period?", "exception",
     ["following the end of the second or fourth periods, the penalty yardage will be enforce"]),
    ("What if the defense has too many players on the field after a substitution?", "exception",
     ["no penalties will be enforced, except for personal fouls and unsportsmanlike conduct fouls, and the down will be replayed"]),
    ("Can the quarterback spike the ball to stop the clock?", "exception",
     ["spiking or throwing the ball in the field of play after a down has ended, except after a touchdown"]),
    ("What is the penalty if a coach throws the red flag with no timeouts left?", "keyword",
     ["Loss of 15 yards enforced as a foul between downs"]),
    ("How many timeouts does each team have in overtime?", "buried",
     ["Each team is entitled to three timeouts during a half"]),
    ("Can a defender grab an opponent's facemask?", "negation",
     ["he must immediately release it. He shall not grasp and control, twist, turn, push, or pull the facem"]),
    ("Can a player be moving toward the line of scrimmage when the ball is snapped?", "negation",
     ["No player is permitted to be moving toward the line of scrimmage when the ball is snapped"]),
    ("Can a team challenge an interception ruling?", "exception",
     ["the play cannot be challenged by either team"]),
    ("Can the spot of the ball be challenged if it is closer to the goal line?", "negation",
     ["The dead ball spot is not reviewable to determine solely whether it should be closer to or further from the goal line"]),
    ("What happens in overtime when the second team kicks a field goal?", "buried",
     ["If B scores a field goal, the game continues in sudden death"]),
    ("Who may recover a fumble on fourth down?", "distractor",
     ["The player who fumbled is the only Team A player permitted to recover and advance the ball prior to a change of possession"]),
    ("Can a fourth down fumble be advanced by the defense?", "distractor",
     ["The ball may be advanced by any member of the defensive team"]),
    ("Is a player allowed to throw a helmet at an opponent?", "negation",
     ["A player may not use a helmet that is no longer worn by anyone as a weapon"]),
    ("What if a defender touches the quarterback after he spikes the ball following the two-minute warning?", "keyword",
     ["spiking or throwing the ball in the field of play after a down has ended, except after a touchdown"]),
    ("Can the offense hurry to snap the ball while a substitution is in progress?", "negation",
     ["the offense is prohibited from rushing quickly to the line of scrimmage and snapping the ball in an obvious attempt"]),
    ("Are tear-away jerseys allowed?", "negation",
     ["Tear-away jerseys are prohibited"]),
    ("Can players roll their pants up above the knee?", "negation",
     ["pants shortened or rolled up are prohibited"]),
    ("Can team staff walk on the field during halftime?", "negation",
     ["Representatives of either team are prohibited from entering the field, unless they are incoming substitutes"]),
    ("Can the head coach go on the field during a game?", "negation",
     ["any game-type activities are prohibited on the field of play"]),
    ("Can a player wear transparent shields on his helmet visor?", "negation",
     ["transparent materials are prohibited"]),
    ("Who loses possession first in overtime if the first team kicks a field goal and the second team fumbles?", "buried",
     ["If the first team to possess the ball scores a field goal or touchdown, after which the second team loses possession by an interception or fumble"]),
]

BI_EVAL = {
    "nfl": FAILURE_SETS["nfl"] + [{"query": q, "category": c, "gold": g} for q, c, g in NFL_PROBES],
    "nba": FAILURE_SETS["nba"],
}
eval_set = BI_EVAL[DOC]
eval_queries = [item["query"] for item in eval_set]
eval_gold = [{c["id"] for c in chunks if is_gold(c["id"], item["gold"])} for item in eval_set]
assert all(eval_gold), "every eval query needs at least one gold chunk"

BGE_PREFIX = "Represent this sentence for searching relevant passages: "
# name -> (HF id, query prefix, document prefix, trust_remote_code)
BI_CANDIDATES = {
    "all-MiniLM-L6-v2 (baseline)": ("sentence-transformers/all-MiniLM-L6-v2", "", "", False),
    "multi-qa-MiniLM-L6-cos-v1":   ("sentence-transformers/multi-qa-MiniLM-L6-cos-v1", "", "", False),
    "bge-small-en-v1.5":           ("BAAI/bge-small-en-v1.5", BGE_PREFIX, "", False),
    "bge-base-en-v1.5":            ("BAAI/bge-base-en-v1.5", BGE_PREFIX, "", False),
    "nomic-embed-text-v1.5":       ("nomic-ai/nomic-embed-text-v1.5", "search_query: ", "search_document: ", True),
}

def _extended_attention_mask(self, attention_mask, input_shape=None, device=None, dtype=None):
    """transformers v4 helper that v5 removed; nomic's remote code still calls it.
    Returns 0 for real tokens and a large negative number for padding."""
    dtype = dtype or self.dtype
    return (1.0 - attention_mask[:, None, None, :].to(dtype)) * torch.finfo(dtype).min


def load_bi_encoder(hf_id, trust):
    model = SentenceTransformer(hf_id, device=DEVICE, trust_remote_code=trust)
    inner = model[0].auto_model
    if trust and not hasattr(inner, "get_extended_attention_mask"):
        type(inner).get_extended_attention_mask = _extended_attention_mask
    return model


chunk_texts = [c["text"] for c in chunks]   # the shared chunks; a document prefix is added only while encoding
rank_cols, summary = {}, []
for name, (hf_id, q_prefix, d_prefix, trust) in BI_CANDIDATES.items():
    try:
        model = load_bi_encoder(hf_id, trust)
    except Exception as e:   # one broken model should not stop the rest of the comparison
        print(f"skipped {name}: {type(e).__name__}: {e}")
        continue
    model.encode(chunk_texts[:8], normalize_embeddings=True)   # warm-up so one-time device setup is not timed
    t = time.perf_counter()
    doc_emb = model.encode([d_prefix + x for x in chunk_texts], normalize_embeddings=True, batch_size=64)
    encode_s = time.perf_counter() - t
    q_emb = model.encode([q_prefix + q for q in eval_queries], normalize_embeddings=True)
    order = np.argsort(-(q_emb @ doc_emb.T), axis=1)   # every chunk ranked by dot product, best first
    best = np.array([int(np.isin(row, list(g)).argmax()) + 1 for row, g in zip(order, eval_gold)])   # best rank of any gold chunk
    rank_cols[name] = best
    summary.append({"model": name, "recall@3": (best <= 3).mean(), "recall@10": (best <= 10).mean(),
                    "MRR@10": np.where(best <= 10, 1 / best, 0).mean(), "encode s": encode_s})
    del model
    if DEVICE == "mps":
        torch.mps.empty_cache()

print(f"(a) Best gold-chunk rank among all {len(chunks)} chunks ({len(eval_queries)} queries, {DOC})")
display(pd.DataFrame(rank_cols, index=eval_queries))
print("\n(b) Summary per model")
display(pd.DataFrame(summary).set_index("model").round({"recall@3": 3, "recall@10": 3, "MRR@10": 3, "encode s": 1}))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14525.36it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7701.05it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

<All keys matched successfully>


(a) Best gold-chunk rank among all 2274 chunks (35 queries, nfl)


,all-MiniLM-L6-v2 (baseline),multi-qa-MiniLM-L6-cos-v1,bge-small-en-v1.5,bge-base-en-v1.5,nomic-embed-text-v1.5
Can a coach challenge a play that begins after the two-minute warning?,7,1,3,17,91
Who is allowed to advance a fumble after the two-minute warning?,10,5,2,1,6
How many challenges does a team get?,6,6,21,6,31
How long is the break between overtime periods?,9,10,8,19,10
Can a team use two charged timeouts back to back?,6,2,3,1,1
Can a kicker wear a detachable kicking toe?,6,70,20,20,2
Which offensive players are not allowed to catch a forward pass?,1,1,2,1,3
What happens when the kickoff lands short of the landing zone?,3,7,1,1,1
What contact is a defender allowed to make on a receiver beyond five yards?,2,2,1,2,4
Is a hip-drop tackle legal?,2,3,1,2,2



(b) Summary per model


,recall@3,recall@10,MRR@10,encode s
model,,,,
all-MiniLM-L6-v2 (baseline),0.514,0.714,0.414,4.2
multi-qa-MiniLM-L6-cos-v1,0.400,0.600,0.352,4.5
bge-small-en-v1.5,0.543,0.657,0.411,8.4
bge-base-en-v1.5,0.486,0.657,0.426,28.2
nomic-embed-text-v1.5,0.514,0.714,0.447,48.1
